In [2]:
import os
import random
import glob
import pyarrow.parquet as pq
import pandas as pd
import subprocess
import time
from dotenv import load_dotenv

load_dotenv()

In [15]:
CURATED_DATA_PATH = os.environ["GPTNL_1_CURATED"]


def get_datasets():
    """
    Get the list of datasets in the CURATED_DATA_PATH directory.
    """
    return [
        d
        for d in os.listdir(CURATED_DATA_PATH)
        if not ((d == "ALL_PARQUET_FILES_VALID") or (d == "dataset_sizes.md"))
    ]


print("Curated datasets found on the system:")
datasets = get_datasets()
for i, d in enumerate(datasets):
    print(f"{i + 1}. {d}")

In [16]:
def show_sample_curated_dataset(dataset_name: str):
    """
    Show a sample of a curated dataset by reading a random parquet file.

    Args:
        dataset_name (str): Name of the dataset
    """

    # Build the path
    path = os.path.join(CURATED_DATA_PATH, dataset_name)
    print(f"Looking for parquet files in: {path}")

    # Find all parquet files
    parquet_files = glob.glob(os.path.join(path, "**", "*.parquet"), recursive=True)
    if not parquet_files:
        raise ValueError(f"No parquet files found in {path}")

    # Select a random parquet file
    random_file = random.choice(parquet_files)
    print(f"Selected file: {random_file}")

    # Read and display the parquet file
    table = pq.read_table(random_file)
    print(f"\nTable shape: {table.num_rows} rows × {table.num_columns} columns")
    print(f"\nTable schema:\n{table.schema}")
    print("\nTable preview:")
    df = table.to_pandas()
    display(df.head())

    # Check if language column exists
    if "language" in df.columns:
        print("\nLanguage column found. Language distribution:")
        lang_counts = df["language"].value_counts()
        lang_percent = df["language"].value_counts(normalize=True) * 100
        lang_df = pd.DataFrame({"count": lang_counts, "percentage": lang_percent})
        display(lang_df)
    else:
        print("\nNo 'language' column found in this dataset.")

    return table


_ = show_sample_curated_dataset("")

Table shape: 85000 rows × 35 columns

Table schema:
text: string
id: string
media: list<element: null>
  child 0, element: null
extra: string
title: string
source: string
author: string
license: string
dataset_name: string
dataset_url: string
dataset_license: string
extraction_timestamp: timestamp[us]
file_path: string
language: string
language_score: double
digit_char_ratio: double
n_char: int64
mean_med_char: double
mean_med_word: double
n_non_symbol_words: int64
avg_word_length: double
hash_ratio: double
ellipsis_ratio: double
bullet_lines_ratio: double
ellipsis_lines_ratio: double
alpha_words_ratio: double
stop_words_count: int64
dup_para_frac: double
dup_para_char_frac: double
dup_line_frac: double
dup_line_char_frac: double
top_n_grams: list<element: list<element: double>>
  child 0, element: list<element: double>
      child 0, element: double
dup_n_grams: list<element: list<element: double>>
  child 0, element: list<element: double>
      child 0, element: double
ToxicLanguage:


Language column found. Language distribution:


,count,percentage
language,,
nl,80501,94.707059
en,4327,5.090588
de,172,0.202353


In [17]:
def get_folder_size_fast(path, method="du", sample_size=50):
    """Calculate the total size of a folder using a faster method
    Args:
        path (str): Path to the folder
        method (str): Method to use ('du', 'sampling', or 'os.walk')
        sample_size (int): Number of files to sample if using 'sampling' method
    Returns:
        float: Size in bytes
    """
    # Method 1: Use 'du' command (Unix/Linux only, fastest)
    if method == "du":
        try:
            start_time = time.time()
            result = subprocess.run(
                ["du", "-sb", path], capture_output=True, text=True, check=True
            )
            size_str = result.stdout.split()[0]  # Format: "size_in_bytes path"
            print(
                f"Folder size calculated with 'du' command in {time.time() - start_time:.2f} seconds"
            )
            return int(size_str)
        except (subprocess.SubprocessError, ValueError, IndexError) as e:
            print(f"Error using 'du' command: {e}. Falling back to sampling method.")
            method = "sampling"  # Fall back to sampling method

    # Method 2: Sample files and extrapolate (faster)
    if method == "sampling":
        start_time = time.time()
        all_files = []

        # Get a list of files (just paths, not sizes yet)
        for dirpath, _, filenames in os.walk(path):
            all_files.extend([os.path.join(dirpath, f) for f in filenames])

        total_files = len(all_files)

        if total_files == 0:
            print(f"No files found in {path}")
            return 0

        # If small number of files, just calculate exactly
        if total_files <= sample_size:
            total_size = sum(
                os.path.getsize(f) for f in all_files if not os.path.islink(f)
            )
            print(
                f"Exact folder size calculated in {time.time() - start_time:.2f} seconds"
            )
            return total_size

        # Sample files and extrapolate
        sampled_files = random.sample(all_files, sample_size)
        sample_size = sum(
            os.path.getsize(f) for f in sampled_files if not os.path.islink(f)
        )
        avg_file_size = sample_size / len(sampled_files)
        estimated_size = avg_file_size * total_files
        print(
            f"Estimated folder size from {len(sampled_files)} sampled files (out of {total_files})"
        )
        print(f"Calculation completed in {time.time() - start_time:.2f} seconds")
        return estimated_size

    # Method 3: Traditional os.walk (slowest but most accurate)
    if method == "os.walk":
        start_time = time.time()
        total_size = 0
        for dirpath, _, filenames in os.walk(path):
            for f in filenames:
                fp = os.path.join(dirpath, f)
                if not os.path.islink(fp):
                    total_size += os.path.getsize(fp)
        print(
            f"Exact folder size calculated with os.walk in {time.time() - start_time:.2f} seconds"
        )
        return total_size

    raise ValueError(f"Unknown method: {method}. Use 'du', 'sampling', or 'os.walk'")


def estimate_overview_curated_data(
    dataset_name, sample_size=200, row_sample_size=200, folder_size_method="du"
):
    """Explore the size of a dataset in various metrics
    Args:
        dataset_name (str): Name of the dataset to analyze
        sample_size (int): Number of parquet files to sample for column statistics
        row_sample_size (int): Number of files to sample for row count estimation
        folder_size_method (str): Method to use for folder size calculation ('du', 'sampling', 'os.walk')
    """

    # Build the path to the dataset
    dataset_path = os.path.join(CURATED_DATA_PATH, dataset_name)
    print(f"Analyzing: {dataset_path}")

    # Metrics to collect
    metrics = {
        "dataset": dataset_name,
        "folder_size_gb": 0,
        "num_parquet_files": 0,
        "total_rows": 0,
        "n_non_symbol_words": 0,
        "n_char": 0,
        "estimated": False,
        "rows_estimated": False,
        "size_estimated": False,
        "languages": {},  # Store language distribution here
    }

    # Calculate folder size efficiently
    print(f"Calculating folder size using {folder_size_method} method...")
    folder_size = get_folder_size_fast(dataset_path, method=folder_size_method)
    metrics["folder_size_gb"] = folder_size / (1024**3)  # Convert bytes to GB
    if folder_size_method == "sampling":
        metrics["size_estimated"] = True

    # Find all parquet files
    print("Finding parquet files...")
    parquet_files = glob.glob(
        os.path.join(dataset_path, "**", "*.parquet"), recursive=True
    )
    metrics["num_parquet_files"] = len(parquet_files)
    print(f"Found {len(parquet_files)} parquet files")

    if not parquet_files:
        print(f"No parquet files found in {dataset_path}")
        return metrics

    # Estimate row count by sampling a subset of files
    actual_row_sample_size = min(row_sample_size, len(parquet_files))
    row_sample_files = random.sample(parquet_files, actual_row_sample_size)

    print(
        f"Estimating rows from {actual_row_sample_size} sampled files (out of {len(parquet_files)})..."
    )
    sample_rows = 0
    for file_path in row_sample_files:
        try:
            parquet_file = pq.ParquetFile(file_path)
            sample_rows += parquet_file.metadata.num_rows
        except Exception as e:
            print(f"Error reading metadata from {file_path}: {e}")
            continue

    # Calculate average rows per file and estimate total
    if actual_row_sample_size > 0:
        avg_rows_per_file = sample_rows / actual_row_sample_size
        metrics["total_rows"] = int(avg_rows_per_file * len(parquet_files))
        if len(parquet_files) > actual_row_sample_size:
            metrics["rows_estimated"] = True
            print(f"Average rows per file: {avg_rows_per_file:.2f}")
            print(f"Estimated total rows: {metrics['total_rows']:,}")
        else:
            print(f"Total rows (exact): {metrics['total_rows']:,}")
    else:
        print("No files sampled for row count")

    # Rest of the function unchanged
    # Sample files for detailed column analysis
    actual_sample_size = min(sample_size, len(parquet_files))

    if len(parquet_files) > actual_sample_size:
        # Try to reuse already sampled files if possible to avoid reading the same file twice
        if actual_sample_size <= actual_row_sample_size:
            sample_files = row_sample_files[:actual_sample_size]
        else:
            # We need more files than we already sampled
            remaining_files = [f for f in parquet_files if f not in row_sample_files]
            additional_files = random.sample(
                remaining_files, actual_sample_size - actual_row_sample_size
            )
            sample_files = row_sample_files + additional_files

        metrics["estimated"] = True
        print(
            f"Sampling {actual_sample_size} out of {len(parquet_files)} files for detailed analysis"
        )
    else:
        sample_files = parquet_files
        print(f"Analyzing all {len(parquet_files)} files for detailed metrics")

    # For each sampled file, check for our columns of interest and calculate stats
    words_per_row_sum = 0
    chars_per_row_sum = 0
    sample_row_count = 0

    print(f"Analyzing {len(sample_files)} sampled files for column statistics...")
    for file_path in sample_files:
        try:
            # First check schema without loading data
            parquet_file = pq.ParquetFile(file_path)
            schema = parquet_file.schema_arrow

            # Build list of columns we can actually read from this file
            available_columns = [field.name for field in schema]
            columns_to_read = []

            if "n_non_symbol_words" in available_columns:
                columns_to_read.append("n_non_symbol_words")

            if "n_char" in available_columns:
                columns_to_read.append("n_char")

            # Only read data if we have columns to analyze
            if columns_to_read:
                table = pq.read_table(file_path, columns=columns_to_read)
                df = table.to_pandas()
                sample_row_count += len(df)

                if "n_non_symbol_words" in df.columns:
                    words_sum = df["n_non_symbol_words"].sum()
                    words_per_row_sum += words_sum

                if "n_char" in df.columns:
                    chars_sum = df["n_char"].sum()
                    chars_per_row_sum += chars_sum
        except Exception as e:
            print(f"Error processing {file_path}: {e}")
            continue

    # If we collected some statistics on sampled files, estimate totals
    if sample_row_count > 0:
        # Calculate averages per row
        avg_words_per_row = (
            words_per_row_sum / sample_row_count if words_per_row_sum > 0 else 0
        )
        avg_chars_per_row = (
            chars_per_row_sum / sample_row_count if chars_per_row_sum > 0 else 0
        )

        # Estimate totals based on the entire dataset row count
        if metrics["estimated"]:
            print(
                f"Estimating metrics based on {sample_row_count:,} sampled rows out of {metrics['total_rows']:,} total rows"
            )
            metrics["n_non_symbol_words"] = avg_words_per_row * metrics["total_rows"]
            metrics["n_char"] = avg_chars_per_row * metrics["total_rows"]
        else:
            metrics["n_non_symbol_words"] = words_per_row_sum
            metrics["n_char"] = chars_per_row_sum

    # Additional code to analyze language distribution in sampled files
    print("Analyzing language distribution in sampled files...")
    language_counts = {}
    total_language_rows = 0

    for file_path in sample_files:
        try:
            # First check schema without loading data
            parquet_file = pq.ParquetFile(file_path)
            schema = parquet_file.schema_arrow

            # Check if language column exists in this file
            available_columns = [field.name for field in schema]
            if "language" in available_columns:
                # Read only the language column to save memory
                table = pq.read_table(file_path, columns=["language"])
                df = table.to_pandas()

                # Count languages
                file_lang_counts = df["language"].value_counts().to_dict()
                total_language_rows += len(df)

                # Add to overall counts
                for lang, count in file_lang_counts.items():
                    if lang in language_counts:
                        language_counts[lang] += count
                    else:
                        language_counts[lang] = count
        except Exception as e:
            print(f"Error processing language in {file_path}: {e}")
            continue

    # Calculate language percentages if we have data
    if total_language_rows > 0:
        language_percentages = {
            lang: (count / total_language_rows) * 100
            for lang, count in language_counts.items()
        }

        # Sort by percentage (descending)
        sorted_percentages = {
            k: v
            for k, v in sorted(
                language_percentages.items(), key=lambda item: item[1], reverse=True
            )
        }

        metrics["languages"] = sorted_percentages

        # Print language distribution
        print("\nLanguage distribution:")
        for lang, pct in sorted_percentages.items():
            print(f"{lang}: {pct:.2f}%")
    else:
        print("No language information found in sampled files")

    # Format numbers for display
    metrics["folder_size_gb"] = round(metrics["folder_size_gb"], 2)
    total_rows_formatted = format(metrics["total_rows"], ",")
    if metrics["rows_estimated"]:
        total_rows_formatted = f"Est. {total_rows_formatted}"

    size_formatted = f"{metrics['folder_size_gb']} GB"
    if metrics["size_estimated"]:
        size_formatted = f"Est. {size_formatted}"

    if metrics["n_non_symbol_words"] > 0:
        n_words_formatted = format(int(metrics["n_non_symbol_words"]), ",")
        est_prefix = "Est. " if metrics["estimated"] else ""
        metrics["n_non_symbol_words_formatted"] = f"{est_prefix}{n_words_formatted}"
    else:
        metrics["n_non_symbol_words_formatted"] = "N/A"

    if metrics["n_char"] > 0:
        n_char_formatted = format(int(metrics["n_char"]), ",")
        est_prefix = "Est. " if metrics["estimated"] else ""
        metrics["n_char_formatted"] = f"{est_prefix}{n_char_formatted}"
    else:
        metrics["n_char_formatted"] = "N/A"

    # Print summary
    print(f"\nDataset: {dataset_name}")
    print(f"Folder size: {size_formatted}")
    print(f"Number of parquet files: {metrics['num_parquet_files']}")
    print(f"Total rows: {total_rows_formatted}")
    print(f"Total non-symbol words: {metrics['n_non_symbol_words_formatted']}")
    print(f"Total characters: {metrics['n_char_formatted']}")

    return metrics

In [18]:
%%time
# Example usage for a single dataset

N_SAMPLES_FOR_STATS = 100
N_SAMPLES_FOR_SIZE = 100

# Run on multiple datasets
all_metrics = []
# Choose a subset of datasets to analyze (or use all)
for dataset_name in datasets:
    try:
        print(f"\n{'=' * 50}\nAnalyzing dataset: {dataset_name}\n{'=' * 50}")
        metrics = estimate_overview_curated_data(
            dataset_name,
            sample_size=N_SAMPLES_FOR_STATS,
            row_sample_size=N_SAMPLES_FOR_SIZE,
            folder_size_method="du",
        )
        if metrics:
            all_metrics.append(metrics)
    except Exception as e:
        print(f"Error analyzing dataset {dataset_name}: {e}")

In [21]:
# Create a DataFrame for comparison
metrics_df = pd.DataFrame(
    [
        {
            "Dataset": m["dataset"],
            "Size (GB)": m["folder_size_gb"],
            "Files": m["num_parquet_files"],
            "Rows": format(m["total_rows"], ",")
            if isinstance(m["total_rows"], int)
            else m["total_rows"],
            "Words (B)": round(m["n_non_symbol_words"] / 1e9, 2)
            if "n_non_symbol_words" in m and m["n_non_symbol_words"] > 0
            else "N/A",
            "Characters (B)": round(m["n_char"] / 1e9, 2)
            if "n_char" in m and m["n_char"] > 0
            else "N/A",
            "Lang.": m["languages"],
            "Est. Tokens (B)": round(m["n_non_symbol_words"] * 1.3 / 1e9, 2)
            if "n_non_symbol_words" in m and m["n_non_symbol_words"] > 0
            else "N/A",
            "Tokens (B) English": round(
                m["n_non_symbol_words"] * 1.3 * m["languages"].get("en", 0) / 100 / 1e9,
                2,
            )
            if "n_non_symbol_words" in m and m["n_non_symbol_words"] > 0
            else 0,
            "Tokens (B) Dutch": round(
                m["n_non_symbol_words"] * 1.3 * m["languages"].get("nl", 0) / 100 / 1e9,
                2,
            )
            if "n_non_symbol_words" in m and m["n_non_symbol_words"] > 0
            else 0,
            "Tokens (B) Other languages": round(
                m["n_non_symbol_words"]
                * 1.3
                * (
                    1
                    - m["languages"].get("en", 0) / 100
                    - m["languages"].get("nl", 0) / 100
                )
                / 1e9,
                2,
            )
            if "n_non_symbol_words" in m and m["n_non_symbol_words"] > 0
            else 0,
        }
        for m in all_metrics
    ]
)

language_df = pd.json_normalize(metrics_df["Lang."]).fillna(0.0)
metrics_df = pd.concat([metrics_df.drop(columns=["Lang."]), language_df], axis=1)

metrics_df

In [ ]:
import yaml

dataset_metadata_yaml = "../dataset_metadata/datasets.yaml"

with open(dataset_metadata_yaml, "r") as f:
    dataset_metadata = yaml.safe_load(f)

for i, k in enumerate(dataset_metadata.keys()):
    print(f"{i + 1} {k}")

In [23]:
name_matching = {
}

name_matching_reversed = {v: k for k, v in name_matching.items()}

evaluated_not_curated = []
curated_not_evaluated = []
curated_and_evaluated = []

for evaluated_dataset in dataset_metadata.keys():
    if evaluated_dataset in name_matching:
        if name_matching[evaluated_dataset] is None:
            evaluated_not_curated.append(evaluated_dataset)
        else:
            curated_and_evaluated.append(evaluated_dataset)
    else:
        print("Dataset not found in name_matching:", evaluated_dataset)

for curated_dataset in datasets:
    if curated_dataset not in name_matching.values():
        curated_not_evaluated.append(curated_dataset)

print("\nEvaluated but not curated datasets:")
for i, dataset in enumerate(evaluated_not_curated):
    print(f"{i + 1}. {dataset}")

print("\nCurated but not evaluated datasets:")
for i, dataset in enumerate(curated_not_evaluated):
    print(f"{i + 1}. {dataset}")

print("\nCurated and evaluated datasets:")
for i, dataset in enumerate(curated_and_evaluated):
    print(f"{i + 1}. {dataset}")
# Check if the datasets are in the curated folder

In [25]:
qualities_and_is_code = {}

code_datasets = [
    "common-corpus-code-filtered",
]

for dataset in curated_and_evaluated:
    quality = dataset_metadata[dataset]["Quality_assurance"]["qualitative_assessment"][
        "score"
    ].lower()
    qualities_and_is_code[name_matching[dataset]] = {
        "quality": quality,
        "is_code": dataset in code_datasets,
    }

# Curated but not evaluated datasets:
# 1. nationaal-archief
# 2. wikidata
# 3. kb
# 4. kb-pd-books
# 5. european-parliament
# 6. kb-open-kranten
# 7. instituut-voor-nl-taal

my_own_estimated_qualities = {
    "american-stories": "low",
    "kb-pd-books": "medium",  # This the label from WP18 listed under Koninklijke Bibliotheek
    "kb-open-kranten": "medium",  # This the label from WP18 listed under Koninklijke Bibliotheek
}

for dataset in curated_not_evaluated:
    if dataset in my_own_estimated_qualities:
        quality = my_own_estimated_qualities[dataset]
        qualities_and_is_code[dataset] = {"quality": quality, "is_code": False}

In [ ]:
for dataset in datasets:
    assert dataset in qualities_and_is_code, (
        f"Dataset {dataset} not found in qualities_and_is_code dictionary"
    )

# Save the qualities to a yaml file
with open("../dataset_metadata/qualities_and_is_code_epoch_1.yaml", "w") as f:
    yaml.dump(qualities_and_is_code, f, default_flow_style=False)
print("Qualities saved to ../dataset_metadata/qualities_and_is_code_epoch_1.yaml")

Qualities saved to ../dataset_metadata/qualities_and_is_code.yaml


In [27]:
# Add the quality column to metrics_df
quality_column = []
is_code_column = []
for dataset in metrics_df["Dataset"]:
    quality_column.append(
        qualities_and_is_code.get(dataset, "unknown").get("quality", "unknown")
    )
    is_code_column.append(
        qualities_and_is_code.get(dataset, "unknown").get("is_code", False)
    )

metrics_df["Quality"] = quality_column
metrics_df["is_code"] = is_code_column

# Reorder columns to put Quality and is_code more in front
# Get all column names
all_columns = metrics_df.columns.tolist()

# Select the columns we want to move to the front
columns_to_front = [
    "Dataset",
    "Quality",
    "is_code",
    "Size (GB)",
    "Files",
    "Rows",
    "Est. Tokens (B)",
    "Tokens (B) English",
    "Tokens (B) Dutch",
    "Tokens (B) Other languages",
]

# Reorder columns
remaining_columns = [col for col in all_columns if col not in columns_to_front]
new_column_order = columns_to_front + remaining_columns
metrics_df = metrics_df[new_column_order]

metrics_df

In [ ]:
# Print the estimated est. tokens in the following buckets
# - dutch_high_quality: Dutch data with high quality
# - dutch_low_medium_quality: Dutch data with low-medium quality
# - english_high_quality: English data with high quality
# - english_low_medium_quality: English data with low-medium quality
# - other_languages: Data in other languages
# - code: Data identified as code


def calculate_token_buckets(df):
    # Define our token buckets
    buckets = {
        "dutch_high_quality": 0,
        "dutch_low_medium_quality": 0,
        "english_high_quality": 0,
        "english_low_medium_quality": 0,
        "other_languages": 0,
        "code": 0,
    }

    # Iterate through rows and calculate token distribution
    for _, row in df.iterrows():
        # Skip rows with N/A token values
        if not isinstance(row["Est. Tokens (B)"], float):
            continue

        # If it's code, add all tokens to code bucket
        if row["is_code"]:
            buckets["code"] += row["Est. Tokens (B)"]
            continue

        # Calculate tokens by quality and language
        is_high_quality = row["Quality"] == "high"

        # Dutch tokens
        dutch_tokens = row["Tokens (B) Dutch"]
        if is_high_quality:
            buckets["dutch_high_quality"] += dutch_tokens
        else:
            buckets["dutch_low_medium_quality"] += dutch_tokens

        # English tokens
        english_tokens = row["Tokens (B) English"]
        if is_high_quality:
            buckets["english_high_quality"] += english_tokens
        else:
            buckets["english_low_medium_quality"] += english_tokens

        # Other language tokens
        buckets["other_languages"] += row["Tokens (B) Other languages"]

    return buckets


# Calculate token buckets
token_buckets = calculate_token_buckets(metrics_df)

# Print the results
print("Estimated tokens (billions) by category:")
for bucket, value in token_buckets.items():
    print(f"- {bucket}: {value:.2f} B")

# Calculate total tokens
total_tokens_sum = sum(token_buckets.values())
print(f"\nTotal tokens: {total_tokens_sum:.2f} B")

# Print percentages
print("\nPercentages:")
for bucket, value in token_buckets.items():
    percentage = (value / total_tokens_sum) * 100
    print(f"- {bucket}: {percentage:.2f}%")

metrics_df.to_csv("../curated_data_overview_epoch_1.csv", index=False)
metrics_df.to_markdown("../curated_data_overview_epoch_1.md", index=False)

total_tokens = sum(
    [t for t in metrics_df["Est. Tokens (B)"].tolist() if isinstance(t, float)]
)
tokal_tokens_en = sum(
    [t for t in metrics_df["Tokens (B) English"].tolist() if isinstance(t, float)]
)
tokal_tokens_nl = sum(
    [t for t in metrics_df["Tokens (B) Dutch"].tolist() if isinstance(t, float)]
)
tokal_tokens_other = sum(
    [
        t
        for t in metrics_df["Tokens (B) Other languages"].tolist()
        if isinstance(t, float)
    ]
)

print()
print(
    f"Total number of tokens (B) in the dataset (estimated): {total_tokens:.2f}",
)
print(
    f"Total number of tokens (B) in the dataset (estimated) (English): {tokal_tokens_en:.2f}"
)
print(
    f"Total number of tokens (B) in the dataset (estimated) (Dutch): {tokal_tokens_nl:.2f}"
)
print(
    f"Total number of tokens (B) in the dataset (estimated) (Other languages): {tokal_tokens_other:.2f}"
)

# Display the updated dataframe
metrics_df

Estimated tokens (billions) by category:
- dutch_high_quality: 11.90 B
- dutch_low_medium_quality: 5.36 B
- english_high_quality: 35.57 B
- english_low_medium_quality: 159.46 B
- other_languages: 32.29 B
- code: 30.93 B

Total tokens: 275.51 B

Percentages:
- dutch_high_quality: 4.32%
- dutch_low_medium_quality: 1.95%
- english_high_quality: 12.91%
- english_low_medium_quality: 57.88%
- other_languages: 11.72%
- code: 11.23%

Total number of tokens (B) in the dataset (estimated): 275.54
Total number of tokens (B) in the dataset (estimated) (English): 224.63
Total number of tokens (B) in the dataset (estimated) (Dutch): 17.27
Total number of tokens (B) in the dataset (estimated) (Other languages): 33.61
